# Faz 3 · B3.1 — Colab ortamı

**Sahip:** Kişi B · **Model:** Qwen2.5-1.5B-Instruct + LoRA (Unsloth)

Bu defter eğitim yapmaz. Eğitimin **koşabileceği ortamı kurar ve kanıtlar.**
Boru hattı provası B3.2'de, gerçek eğitim B3.3'te.

---

## Ücretsiz Colab'ın üç kısıtı

Bunlar aksaklık değil, **tasarımı belirleyen şartlar**:

| kısıt | sonucu |
|---|---|
| Tek T4 GPU | ~55k örnek için 4-7 saat |
| Oturum ~90 dk hareketsizlikte, ~12 saatte kesin kopar | 4-7 saatlik tek koşu **büyük olasılıkla yarıda kesilir** |
| `/content` uçucu | Oturum kapanınca dosyalar gider |

Bu yüzden eğitim **tek koşu değil, kaldığı yerden devam edebilen** bir süreç
olarak kurulacak ve her şey **Drive'da** duracak. Tercih değil, zorunluluk.


## 1. GPU var mı

`Runtime → Change runtime type → T4 GPU` seçili olmalı.


In [ ]:
!nvidia-smi


In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU yok! Runtime > Change runtime type > T4 GPU'

ad = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU  : {ad}')
print(f'VRAM : {vram:.1f} GB')
if 'T4' not in ad:
    print(f'{chr(10)}NOT: T4 bekleniyordu, {ad} bulundu. Sureler farkli cikar.')


## 2. Drive

Her Colab oturumu sıfırdan başlar; bu hücre **her açılışta** çalıştırılmalı.
Açılan pencereden Google hesabını seç ve izin ver.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 3. Klasör düzeni

```
/content/drive/MyDrive/codifya/
  veri/        <- Kisi A'nin JSONL dosyalari (BIR KEZ yuklenir)
  checkpoint/  <- egitim ara kayitlari (oturum kopunca buradan devam)
  cikti/       <- merge edilmis model + GGUF
```

> ⚠️ **Veriyi her oturumda yeniden yükleme.** 76 MB'lık `gerekce_train.jsonl`
> her oturumda 10-20 dakika yakar ve kopma riskini artırır. Drive'a bir kez koy.


In [ ]:
from pathlib import Path

KOK = Path('/content/drive/MyDrive/codifya')
VERI = KOK / 'veri'
CHECKPOINT = KOK / 'checkpoint'
CIKTI = KOK / 'cikti'

for d in (VERI, CHECKPOINT, CIKTI):
    d.mkdir(parents=True, exist_ok=True)
    print(f'{d!s:45s} hazir')


## 4. Veri yerinde mi

Dosyalar Drive'daki `codifya/veri/` klasörüne elle yüklenir (Drive arayüzünden
sürükle-bırak). Aşağıdaki hücre eksik olanı söyler.


In [ ]:
BEKLENEN = {
    'gerekce_train.jsonl': 40293,
    'gerekce_val.jsonl': None,
    'gerekce_test.jsonl': None,
    'router_train.jsonl': 43798,
    'router_val.jsonl': None,
    'router_test.jsonl': None,
    'golden_set_aday.jsonl': None,
}

eksik = []
for ad_, beklenen in BEKLENEN.items():
    yol = VERI / ad_
    if not yol.exists():
        eksik.append(ad_)
        print(f'  YOK  {ad_}')
        continue
    mb = yol.stat().st_size / 1024**2
    with yol.open(encoding='utf-8') as f:
        n = sum(1 for _ in f)
    uyari = ''
    if beklenen and n != beklenen:
        uyari = f'  <-- {beklenen} bekleniyordu'
    print(f'  VAR  {ad_:24s} {mb:7.1f} MB  {n:6d} satir{uyari}')

if eksik:
    raise SystemExit(f'Eksik dosya: {eksik} -- Drive/codifya/veri/ klasorune yukle.')
print('Veri tamam.')


## 5. Unsloth

Kurulum 2-3 dakika sürer ve **her oturumda tekrarlanır** (Colab sıfırlanıyor).


In [ ]:
!pip install -q unsloth


In [ ]:
from unsloth import FastLanguageModel

print('unsloth hazir:', FastLanguageModel)


## 6. ⭐ İstem biçimi — eğitimden ÖNCE karara bağlandı

Yanlış karar 5 saatlik eğitimi çöpe atar. Bu yüzden biçim **yerelde** kararlaştırıldı,
veri de yerelde dönüştürüldü. Colab'da dönüştürme adımı yok — yüklenen dosyalar
doğrudan eğitime hazır:

```json
{"istem": "VERILER:\nurun: ...\n\nGEREKCE:", "cevap": "Porselen Karo ..."}
```

Dönüştürmeyi yapan betik: `scratchpad/veri_hazirla.py`. Yüklenecek boyut
**103 MB → 34 MB** düştü ve Colab'da patlayacak bir adım azaldı.

---

### Karar 1 — ürün adı isteme KONUYOR

```
Egitim verisi (Kisi A) : gerekcelerin %100'unde urun adi AYNEN geciyor
B2.4 istemi   (Kisi B) : urun adini BILEREK koymuyordu
```

B2.4'te adı çıkarmıştım çünkü **taban model** Türkçe özel adları bozuyordu:
`Astar Boya` → *starboy*, `İzocam` → *isyancı yalıtım levhası*. Guard bunu
yakalayamaz — uydurulan şey sayı değil.

Eğitimde istem ile hedef arasındaki ilişki neyse, model onu öğrenir:

- **İsteme ad koymaz, hedefte ad varsa** → model *yoktan ad uydurmayı* öğrenir.
  "starboy" sorununu ağırlıklara işler. **En kötü seçenek.**
- **İsteme ad koyar, hedefte ad varsa** → model *adı kopyalamayı* öğrenir.
  Taban model beceremiyordu çünkü hiç öğretilmemişti.

Kopyalama davranışı **B3.2'de 100 örnekle sınanacak** — tam eğitimden önce, ucuz.
Model yine ad bozarsa geri dönüş planı: hedeflerden adları ayıklamak (bir kerelik
dönüşüm, Kişi A'nın veriyi yeniden üretmesi gerekmez).

### Karar 2 — eğitilmiş modelin istemi KISA

B2.4'ün istemi taban model içindi; kurallar bloğu ve few-shot örnek içeriyordu.
Eğitilmiş modelde ikisi de gereksiz — davranış ağırlıklara işlenmiş olacak.
Kısa istem = az token = hızlı üretim.

### Guard düzeltmesi (eğitimi doğrudan etkiledi)

Veriyi incelerken B2.5 guard'ında bir hata bulundu: %2 bağıl sınır, küçük
sayıların 2 ondalıkta **doğru** yuvarlanmasını da kesiyordu
(`0,14444 → "0,14"`, bağıl fark %3,08). Bu, sipariş gerekçelerinin **%16,9'unu**
boşuna reddediyordu. Düzeltildi; artık 5.000 hedefin 5.000'i guard'dan geçiyor.


In [ ]:
import json

with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
    ornek = json.loads(f.readline())

print('--- ISTEM ---')
print(ornek['istem'])
print()
print('--- CEVAP (hedef) ---')
print(ornek['cevap'])

assert set(ornek) >= {'istem', 'cevap'}, 'Dosya hazir bicimde degil!'
print()
print('Bicim dogru: istem + cevap')


## 7. Bitti sayılır — üç ölçüt

Üçü de yeşilse B3.2'ye geçilir.


In [ ]:
olcutler = []

try:
    olcutler.append((torch.cuda.is_available(), 'GPU baglandi: ' + torch.cuda.get_device_name(0)))
except Exception as e:
    olcutler.append((False, 'GPU yok: ' + str(e)))

try:
    from unsloth import FastLanguageModel

    olcutler.append((KOK.exists(), 'Drive bagli: ' + str(KOK)))
    olcutler.append((True, 'unsloth import edildi'))
except Exception as e:
    olcutler.append((False, 'unsloth: ' + str(e)))

try:
    with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
        k = json.loads(f.readline())
    tamam = bool(k.get('istem')) and bool(k.get('cevap'))
    olcutler.append((tamam, 'veri/ okunabiliyor ve bicim dogru'))
except Exception as e:
    olcutler.append((False, 'veri okunamadi: ' + str(e)))

print('=' * 60)
for ok, mesaj in olcutler:
    print('  ' + ('GECTI' if ok else 'KALDI') + '  ' + mesaj)
print('=' * 60)
print('B3.1 TAMAM' if all(o for o, _ in olcutler) else 'B3.1 EKSIK')


---

# B3.2 · 100 örnekle boru hattı provası

**Bu adım atlanmaz.** Görev dosyasının gerekçesi:

> Tam eğitim 4-7 saat sürüyor ve ücretsiz Colab'da oturum kopabiliyor.
> Hattın 3. saatinde çıkacak bir biçim hatası, hem 3 saati hem bir Colab
> oturumunu yakar.

Amaç **kalite değil, hattın çalıştığını kanıtlamak**. 100 örnekle eğitilen
model kötü olacak — sorun değil. Kanıtlanmak istenen şu zincir:

```
yukle -> egit -> kaydet -> uret -> (B3.4'te) merge -> GGUF -> Ollama
```

### Burada ayrıca sınanan karar

B3.1'de kararlaştırdık: **ürün adı isteme konuyor**, model kopyalamayı
öğrensin diye. Taban model bunu beceremiyordu (`Astar Boya` → *starboy*).
100 örnek öğrenmeye yetmez ama **eğilimi** gösterir.


## 1. Taban modeli yükle


In [ ]:
from unsloth import FastLanguageModel

MODEL = 'unsloth/Qwen2.5-1.5B-Instruct'
MAX_SEQ = 1024  # gerekce istemleri kisa; daha uzunu bosa VRAM ve sure yakar

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)
print('yuklendi:', MODEL)


## 2. LoRA katmanlarını tak

`r=16`, `alpha=32`, hedef: tüm dikkat + MLP projeksiyonları (görev dosyası).
Eğitilen parametre oranı %1'in altında kalmalı — LoRA'nın bütün amacı bu.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model.print_trainable_parameters()


## 3. 100 örnek hazırla

Eğitim metni `istem + cevap + EOS` olarak birleştiriliyor. **EOS şart**:
olmazsa model cümleyi nerede bitireceğini öğrenmez ve `max_new_tokens`
dolana kadar saçmalar.


In [ ]:
import json

from datasets import Dataset

N = 100

with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
    kayitlar = [json.loads(satir) for _, satir in zip(range(N), f, strict=False)]

metinler = [k['istem'] + '\n' + k['cevap'] + tokenizer.eos_token for k in kayitlar]
ds = Dataset.from_dict({'text': metinler})

print(len(ds), 'ornek hazir')
print('-' * 60)
print(ds[0]['text'])


## 4. Eğit (~2 dakika)

`max_steps=30` bilinçli: amaç öğrenmek değil, hattın çalıştığını görmek.
Kayıp (`loss`) düşüyorsa eğitim gerçekten oluyor demektir.


In [ ]:
from trl import SFTTrainer

ARGS = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=30,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    output_dir=str(CHECKPOINT / 'b32'),
    report_to='none',
)

# TRL surumleri arasinda imza degisti; ikisini de deniyoruz.
try:
    from trl import SFTConfig

    trainer = SFTTrainer(
        model=model,
        train_dataset=ds,
        args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_SEQ, **ARGS),
    )
except (ImportError, TypeError) as e:
    print('SFTConfig kullanilamadi, eski imzaya duseliyor:', type(e).__name__)
    from transformers import TrainingArguments

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ,
        args=TrainingArguments(**ARGS),
    )

sonuc = trainer.train()
print(sonuc)


## 5. ⭐ Üret ve oku — asıl sınav

Aynı istemi hem **eğitim öncesi hedefle** hem modelin ürettiğiyle
karşılaştırıyoruz. Bakılacak üç şey:

1. Model `GEREKCE:`den sonra **cümle** kuruyor mu (istem tekrarı değil)?
2. **Ürün adını doğru kopyalıyor mu** — yoksa yine *starboy* mu?
3. Cümle bitiyor mu (EOS öğrenilmiş mi)?


In [ ]:
FastLanguageModel.for_inference(model)

for i in (0, 1, 2):
    istem = kayitlar[i]['istem']
    girdi = tokenizer([istem], return_tensors='pt').to('cuda')
    cikti = model.generate(**girdi, max_new_tokens=120, do_sample=False)
    tam = tokenizer.decode(cikti[0], skip_special_tokens=True)
    uretilen = tam[len(istem):].strip()
    ad = kayitlar[i]['istem'].split('urun: ')[1].split(chr(10))[0]

    print('=' * 70)
    print('URUN   :', ad)
    print('URETTI :', uretilen[:300])
    print('HEDEF  :', kayitlar[i]['cevap'][:300])
    print('ad dogru kopyalandi mi :', ad in uretilen)


## 6. LoRA'yı Drive'a kaydet

Yalnızca adaptör kaydediliyor (~50 MB), tam model değil (~3 GB). B3.4'te
taban modelle birleştirilip GGUF'a çevrilecek.


In [ ]:
HEDEF_DIZIN = CIKTI / 'b32_lora'

model.save_pretrained(str(HEDEF_DIZIN))
tokenizer.save_pretrained(str(HEDEF_DIZIN))

import os

toplam = sum(os.path.getsize(os.path.join(k, d)) for k, _, dl in os.walk(HEDEF_DIZIN) for d in dl)
print('kaydedildi:', HEDEF_DIZIN)
print('boyut     :', round(toplam / 1024**2, 1), 'MB')
print('dosyalar  :', sorted(os.listdir(HEDEF_DIZIN)))


## 7. B3.2 bitti sayılır


In [ ]:
olcutler = [
    (len(ds) == N, str(N) + ' ornek yuklendi'),
    (sonuc.training_loss > 0, 'egitim kostu, kayip: ' + str(round(sonuc.training_loss, 4))),
    (HEDEF_DIZIN.exists(), 'LoRA Drive\'a kaydedildi'),
    (bool(uretilen), 'model metin uretti'),
]

print('=' * 60)
for ok, mesaj in olcutler:
    print('  ' + ('GECTI' if ok else 'KALDI') + '  ' + mesaj)
print('=' * 60)
print('B3.2 TAMAM' if all(o for o, _ in olcutler) else 'B3.2 EKSIK')
print()
print('NOT: kalite BEKLENMIYOR. 100 ornek ogrenmeye yetmez.')
print('     Kanitlanan sey hattin ucdan uca calistigi.')


## 8. ⭐ Görülmemiş örnekle sınav — ezber mi, öğrenme mi?

İlk ölçümde üç örnekte de ürün adı doğru kopyalandı **ama üretilen metin
hedefle harfi harfine aynı çıktı.** Model o üç örneği ezberlemiş; 100 örnek
ve 30 adımda beklenen şey bu.

Yani şu ana kadar cevaplanan soru *"kopyalayabiliyor mu"* değil,
*"ezberleyebiliyor mu"* idi. Gerçek cevap için modelin **hiç görmediği**
bir örnek gerekiyor — `gerekce_val.jsonl` tam bunun için ayrılmış.

⚠️ Beklenti düşük tutulmalı: 100 örnek genelleme için yetmez. Buradan
çıkacak sonuç bir **işaret**, kanıt değil. Asıl karar B3.5'te, tam eğitim
sonrası verilecek.


In [ ]:
import json

with (VERI / 'gerekce_val.jsonl').open(encoding='utf-8') as f:
    gorulmemis = [json.loads(satir) for _, satir in zip(range(5), f, strict=False)]

dogru = 0
for k in gorulmemis:
    istem = k['istem']
    ad = istem.split('urun: ')[1].split(chr(10))[0]
    girdi = tokenizer([istem], return_tensors='pt').to('cuda')
    cikti = model.generate(**girdi, max_new_tokens=120, do_sample=False)
    uretilen = tokenizer.decode(cikti[0], skip_special_tokens=True)[len(istem):].strip()
    kopyalandi = ad in uretilen
    dogru += kopyalandi
    ezber = uretilen.strip() == k['cevap'].strip()

    print('=' * 70)
    print('URUN        :', ad)
    print('ad kopyalandi:', kopyalandi)
    print('hedefle ayni :', ezber, ' <- True ise ezber (val setinde olmamali)')
    print('URETTI      :', uretilen[:250])
    print('HEDEF       :', k['cevap'][:250])

print()
print('=' * 70)
print('Gorulmemis ornekte ad kopyalama:', dogru, '/', len(gorulmemis))
print('NOT: 100 ornek genelleme icin yetmez. Bu bir ISARET, kanit degil.')


---

# B3.3 · Tam LoRA eğitimi

⚠️ **Bu bölüm B3.2'den bağımsız çalışır.** Taban modeli sıfırdan yükler —
B3.2'de 100 örnekle eğitilmiş ağırlıklarla karışmasın diye. Oturum
koptuğunda 1-5. hücreleri (Drive + unsloth) ve buradan itibarını çalıştır.

## Üç tasarım kararı, üçü de görev dosyasından

**1. Router ve gerekçe TEK modelde.** İki ayrı model yönetmek gereksiz yük.
Ayrım `GOREV:` etiketiyle yapılıyor — modelin ilk gördüğü satır bu.

**2. Oturum kopması hata değil, beklenen durum.** Checkpoint 200 adımda bir
Drive'a yazılıyor, `save_total_limit=2` ile Drive dolmuyor. Kopunca
`resume_from_checkpoint=True` kaldığı yerden devam ettiriyor.

**3. Küçükten büyüğe.** 55 bin örnekle başlanmıyor:

| tur | örnek | süre | amaç |
|---|---|---|---|
| **1** | ~15k | ~1,5 saat | Tek oturumda biter. İyileşme var mı? |
| 2 | ~35k | ~3-4 saat | 1. turdan iyiyse |
| 3 | ~55k | ~5-7 saat | Ancak 2. tur iyileşme gösterdiyse |

> 1. turdan 2. tura kayda değer iyileşme yoksa darboğaz **veri miktarı**
> değil, **veri kalitesi**. O zaman daha uzun beklemek çözüm değil.

Bugün **1. tur** koşuyor.


## 1. İki görevi tek biçimde birleştir

`GOREV:` etiketi modele hangi işi yaptığını söylüyor. Etiket olmadan model
iki davranışı karıştırır — sipariş sorusuna gerekçe, gerekçe istemine araç
adı yazar.

⚠️ Bu biçim Faz 4'te çalışma zamanına da taşınmalı: `app/llm/router.py` ve
`app/llm/explain.py` aynı `GOREV:` başlığını kullanmazsa eğitilmiş model
tanımadığı bir istem görür.


In [ ]:
import json
import random

from datasets import Dataset

TUR_ORNEK = 15_000  # 1. tur
random.seed(42)


def satirlari_oku(ad, sinir=None):
    with (VERI / ad).open(encoding='utf-8') as f:
        return [json.loads(s) for s in (f if sinir is None else [x for _, x in zip(range(sinir), f, strict=False)])]


def gerekce_metni(k):
    return 'GOREV: gerekce' + chr(10) + k['istem'] + chr(10) + k['cevap']


def router_metni(k):
    cevap = json.dumps(
        {'arac': k['arac'], 'parametreler': k.get('parametreler', {})},
        ensure_ascii=False,
    )
    return 'GOREV: router' + chr(10) + 'SORU: ' + k['soru'] + chr(10) * 2 + 'ARAC:' + chr(10) + cevap


yari = TUR_ORNEK // 2
g = [gerekce_metni(k) for k in satirlari_oku('gerekce_train.jsonl', yari)]
r = [router_metni(k) for k in satirlari_oku('router_train.jsonl', yari)]

metinler = g + r
random.shuffle(metinler)
metinler = [m + tokenizer.eos_token for m in metinler]

egitim = Dataset.from_dict({'text': metinler})
print('gerekce:', len(g), ' router:', len(r), ' toplam:', len(egitim))
print('-' * 70)
print(egitim[0]['text'][:400])
print('-' * 70)
print(egitim[1]['text'][:400])


## 2. Doğrulama kümesi

**Bitti sayılır** ölçütü buna bakıyor: doğrulama kaybı düşüp yatay seyre
geçmeli. Eğitim kaybı düşerken doğrulama kaybı **yükselmeye başlarsa** model
ezberliyor demektir — orada durulur.

Küçük tutuluyor (500+500): her değerlendirme eğitimi duraklatıyor.


In [ ]:
gv = [gerekce_metni(k) for k in satirlari_oku('gerekce_val.jsonl', 500)]
rv = [router_metni(k) for k in satirlari_oku('router_val.jsonl', 500)]

dogrulama = Dataset.from_dict({'text': [m + tokenizer.eos_token for m in gv + rv]})
print('dogrulama ornegi:', len(dogrulama))


## 3. Taban modeli SIFIRDAN yükle

B3.2'nin 100 örnekle eğitilmiş ağırlıkları karışmasın diye yeniden yükleniyor.
Oturum yeni açıldıysa bu zaten gerekli.


In [ ]:
from unsloth import FastLanguageModel

MODEL = 'unsloth/Qwen2.5-1.5B-Instruct'
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model.print_trainable_parameters()


## 4. Eğitici — checkpoint Drive'a, sık

`save_steps=200` (~7 dakikada bir) ve `save_total_limit=2`. Kopma her an
olabilir; en fazla 7 dakikalık iş kaybedilir.


In [ ]:
from trl import SFTTrainer

CIKIS = str(CHECKPOINT / 'b33_tur1')

ARGS = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    output_dir=CIKIS,
    report_to='none',
    save_steps=200,
    save_total_limit=2,
    eval_strategy='steps',
    eval_steps=200,
)


def egitici_kur():
    try:
        from trl import SFTConfig

        return SFTTrainer(
            model=model,
            train_dataset=egitim,
            eval_dataset=dogrulama,
            args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_SEQ, **ARGS),
        )
    except (ImportError, TypeError) as e:
        print('SFTConfig kullanilamadi:', type(e).__name__)
        from transformers import TrainingArguments

        return SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=egitim,
            eval_dataset=dogrulama,
            dataset_text_field='text',
            max_seq_length=MAX_SEQ,
            args=TrainingArguments(**ARGS),
        )


trainer = egitici_kur()
adim = len(egitim) // (2 * 4)
print('toplam adim (1 epoch):', adim)
print('tahmini sure          :', round(adim * 2.1 / 60), 'dakika')


## 5. ⭐ ÖNCE devam-etme testini yap

Görev dosyası bunu özellikle istiyor:

> Bu satır olmadan her kopma seni başa döndürür. **Önce bunun çalıştığını
> doğrula.** Bu 15 dakikalık test, sonraki günlerde saatler kazandırır.

Elle kesmek yerine programla yapıyoruz: 40 adım koş → dur → checkpoint'ten
devam et → sayacın 40'tan devam ettiğini gör.

Sayaç 0'dan başlarsa **devam etme çalışmıyor** demektir ve uzun eğitime
girmenin anlamı yok.


In [ ]:
import shutil

TEST_CIKIS = str(CHECKPOINT / 'b33_devam_testi')
shutil.rmtree(TEST_CIKIS, ignore_errors=True)

TEST_ARGS = {**ARGS, 'output_dir': TEST_CIKIS, 'max_steps': 40, 'save_steps': 20,
             'eval_strategy': 'no', 'num_train_epochs': 1, 'logging_steps': 20}


def test_egitici(args):
    try:
        from trl import SFTConfig

        return SFTTrainer(model=model, train_dataset=egitim,
                          args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_SEQ, **args))
    except (ImportError, TypeError):
        from transformers import TrainingArguments

        return SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=egitim,
                          dataset_text_field='text', max_seq_length=MAX_SEQ,
                          args=TrainingArguments(**args))


t1 = test_egitici(TEST_ARGS)
s1 = t1.train()
print('1. kosu bitti, adim:', s1.global_step)


In [ ]:
import os

kayitlar = sorted(d for d in os.listdir(TEST_CIKIS) if d.startswith('checkpoint-'))
print('Drive\'daki checkpointler:', kayitlar)

t2 = test_egitici({**TEST_ARGS, 'max_steps': 80})
s2 = t2.train(resume_from_checkpoint=True)

print()
print('=' * 60)
print('1. kosu bitis adimi :', s1.global_step)
print('2. kosu bitis adimi :', s2.global_step)
gecti = s2.global_step > s1.global_step and len(kayitlar) > 0
print('DEVAM ETME:', 'CALISIYOR' if gecti else 'CALISMIYOR -- uzun egitime girme!')
print('=' * 60)


## 6. 1. tur eğitimi (~1,5 saat)

⚠️ **Sekmeyi kapatma, bilgisayarı uyutma.** Ücretsiz Colab ~90 dakika
hareketsizlikte oturumu kapatır.

Kopma olursa panik yok: defteri yeniden aç, Drive + unsloth hücrelerini ve
B3.3'ün 1-4. hücrelerini çalıştır, sonra aşağıdaki hücreyi
`resume_from_checkpoint=True` ile tekrar koştur.


In [ ]:
import os

devam = os.path.isdir(CIKIS) and any(d.startswith('checkpoint-') for d in os.listdir(CIKIS))
print('onceki checkpoint var mi:', devam)

sonuc = trainer.train(resume_from_checkpoint=devam)
print(sonuc)


## 7. Kayıp eğrisi — ezberliyor mu?

Bakılacak şey: **doğrulama kaybı düşüp yatay seyre geçti mi?**
Eğitim kaybı düşerken doğrulama kaybı yükseliyorsa model ezberliyor.


In [ ]:
kayit = trainer.state.log_history
egitim_kaybi = [(k['step'], k['loss']) for k in kayit if 'loss' in k]
dogrulama_kaybi = [(k['step'], k['eval_loss']) for k in kayit if 'eval_loss' in k]

print('adim   egitim   dogrulama')
print('-' * 32)
d = dict(dogrulama_kaybi)
for adim_, kayip in egitim_kaybi:
    dk = d.get(adim_)
    print(f'{adim_:5d}  {kayip:7.4f}  {dk if dk is None else round(dk, 4)}')

if len(dogrulama_kaybi) >= 2:
    ilk, son = dogrulama_kaybi[0][1], dogrulama_kaybi[-1][1]
    en_iyi = min(k for _, k in dogrulama_kaybi)
    print()
    print('ilk dogrulama kaybi :', round(ilk, 4))
    print('son dogrulama kaybi :', round(son, 4))
    print('en iyi              :', round(en_iyi, 4))
    print('EZBERLEME UYARISI' if son > en_iyi * 1.05 else 'Ezberleme isareti yok')


## 8. LoRA'yı Drive'a kaydet


In [ ]:
TUR1 = CIKTI / 'b33_tur1_lora'

model.save_pretrained(str(TUR1))
tokenizer.save_pretrained(str(TUR1))

import os

boyut = sum(os.path.getsize(os.path.join(k, d)) for k, _, dl in os.walk(TUR1) for d in dl)
print('kaydedildi:', TUR1)
print('boyut     :', round(boyut / 1024**2, 1), 'MB')


---

# B3.5 (1. tur) · Router ölçümü

**B2.3'ün birebir aynı 30 soruluk seti.** Set deftere gömülü — aynı sorular,
aynı puanlama, aynı kabul kuralları. Tek değişen model.

| | taban çizgi (B2.3) | 1. tur |
|---|---|---|
| model | `qwen2.5:1.5b-instruct` ham | + LoRA (15k örnek) |
| araç doğruluğu | **%70,0** | ? |
| araç + parametre | **%66,7** | ? |
| `olu_stok_sorgula` | **1/5** | ? |

### Puanlama kuralları (B2.3'ten aynen)

1. **Parametre biçimi esnek** — ürün adı da SKU kodu da kabul. Model hangisini
   yazarsa yazsın ölçüm geçerli.
2. **Büyük/küçük harf duyarsız** — "kirmizi tugla" ile "Kırmızı Tuğla" farkı
   bu ölçümün konusu değil.
3. **Uydurma parametre sayılır** — soruda geçmeyen bir değer üretmek, araç
   doğru olsa bile kalite sorunu.
4. **Çöküş dedektörü** — model tek araca çökerse (pay ≥ %40) doğruluk yüksek
   çıksa bile uyarı verir.

Üretim **açgözlü** (`do_sample=False`), tıpkı taban çizgide olduğu gibi —
aynı sorunun iki koşuda farklı cevap vermesi ölçümü geçersiz kılardı.


In [ ]:
SORULAR = [
    {"soru": "Kritik stok seviyesine düşen ürünler hangileri?", "arac": "kritik_stok_sorgula", "parametreler": {}, "stil": "acik"},
    {"soru": "Boya kategorisinde kritik stok var mı?", "arac": "kritik_stok_sorgula", "parametreler": {"kategori": ["Boya", "boya"]}, "stil": "acik"},
    {"soru": "Hangi ürünler tükenmek üzere?", "arac": "kritik_stok_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "Neyin acilen sipariş edilmesi gerekiyor?", "arac": "kritik_stok_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "stok azalan var mi", "arac": "kritik_stok_sorgula", "parametreler": {}, "stil": "gunluk"},
    {"soru": "Ölü stok durumundaki ürünleri listeler misin?", "arac": "olu_stok_sorgula", "parametreler": {}, "stil": "acik"},
    {"soru": "Çimento kategorisinde hareketsiz ürün var mı?", "arac": "olu_stok_sorgula", "parametreler": {"kategori": ["Çimento", "cimento", "çimento"]}, "stil": "acik"},
    {"soru": "Aylardır dönmeyen mal var mı elimizde?", "arac": "olu_stok_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "Rafta bekleyip duran ürünleri görebilir miyim?", "arac": "olu_stok_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "tasfiye onerlen urun var mi", "arac": "olu_stok_sorgula", "parametreler": {}, "stil": "gunluk"},
    {"soru": "T-0014 tedarikçisinin performansı nedir?", "arac": "tedarikci_performansi_sorgula", "parametreler": {"tedarikci_id": ["T-0014", "t-0014"]}, "stil": "acik"},
    {"soru": "Yılmaz Yapı'nın teslimat skoru kaç?", "arac": "tedarikci_performansi_sorgula", "parametreler": {"tedarikci_id": ["Yılmaz Yapı", "Yılmaz Yapı'nın"]}, "stil": "acik"},
    {"soru": "Bizi kim geciktiriyor?", "arac": "tedarikci_performansi_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "T-0031 güvenilir mi?", "arac": "tedarikci_performansi_sorgula", "parametreler": {"tedarikci_id": ["T-0031", "t-0031"]}, "stil": "dolayli"},
    {"soru": "t-0007 gecikiyomu", "arac": "tedarikci_performansi_sorgula", "parametreler": {"tedarikci_id": ["T-0007", "t-0007"]}, "stil": "gunluk"},
    {"soru": "Kırmızı Tuğla (S-01432) için sipariş önerisi nedir?", "arac": "siparis_onerisi_sorgula", "parametreler": {"sku_id": ["Kırmızı Tuğla", "S-01432"]}, "stil": "acik"},
    {"soru": "Alçıpan (S-00318) için ne kadar sipariş verilmeli?", "arac": "siparis_onerisi_sorgula", "parametreler": {"sku_id": ["Alçıpan", "S-00318"]}, "stil": "acik"},
    {"soru": "Portland Çimento (S-01949) ürününden kaç tane almalıyım?", "arac": "siparis_onerisi_sorgula", "parametreler": {"sku_id": ["Portland Çimento", "S-01949"]}, "stil": "dolayli"},
    {"soru": "Demir profil (S-01971) için ne ısmarlayalım?", "arac": "siparis_onerisi_sorgula", "parametreler": {"sku_id": ["Demir profil", "S-01971"]}, "stil": "dolayli"},
    {"soru": "kirmizi tugla (S-01432) kac adet siparis verelim", "arac": "siparis_onerisi_sorgula", "parametreler": {"sku_id": ["Kırmızı Tuğla", "kirmizi tugla", "S-01432"]}, "stil": "gunluk"},
    {"soru": "Onay bekleyen kararlar var mı?", "arac": "onay_kuyrugu_sorgula", "parametreler": {}, "stil": "acik"},
    {"soru": "Onay kuyruğunu gösterir misin?", "arac": "onay_kuyrugu_sorgula", "parametreler": {}, "stil": "acik"},
    {"soru": "Benim onaylamam gereken bir şey var mı?", "arac": "onay_kuyrugu_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "Sistem neyi bana sordu?", "arac": "onay_kuyrugu_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "bekleyen karar var mi", "arac": "onay_kuyrugu_sorgula", "parametreler": {}, "stil": "gunluk"},
    {"soru": "Gecelik özeti gösterir misin?", "arac": "gecelik_ozet_sorgula", "parametreler": {}, "stil": "acik"},
    {"soru": "Dün gece sistem ne buldu?", "arac": "gecelik_ozet_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "Sabah raporunda ne çıkmış?", "arac": "gecelik_ozet_sorgula", "parametreler": {}, "stil": "dolayli"},
    {"soru": "Genel stok durumu nedir?", "arac": "genel_stok_durumu_sorgula", "parametreler": {}, "stil": "acik"},
    {"soru": "Depoda genel olarak durum ne?", "arac": "genel_stok_durumu_sorgula", "parametreler": {}, "stil": "dolayli"},
]

print(len(SORULAR), 'soru yuklendi')


In [ ]:
import json
import re
import time
from collections import Counter

TABAN = {"arac_dogruluk": 0.7, "tam_dogruluk": 0.6666666666666666, "arac_bazinda": {"gecelik_ozet_sorgula": [3, 3], "genel_stok_durumu_sorgula": [2, 2], "kritik_stok_sorgula": [3, 5], "olu_stok_sorgula": [1, 5], "onay_kuyrugu_sorgula": [4, 5], "siparis_onerisi_sorgula": [4, 5], "tedarikci_performansi_sorgula": [4, 5]}, "stil_bazinda": {"acik": [9, 12], "dolayli": [9, 13], "gunluk": [3, 5]}}

FastLanguageModel.for_inference(model)


def cevap_uret(soru):
    istem = 'GOREV: router' + chr(10) + 'SORU: ' + soru + chr(10) * 2 + 'ARAC:'
    girdi = tokenizer([istem], return_tensors='pt').to('cuda')
    cikti = model.generate(**girdi, max_new_tokens=80, do_sample=False)
    tam = tokenizer.decode(cikti[0], skip_special_tokens=True)
    return tam[len(istem):].strip()


def json_ayikla(metin):
    m = re.search(r'\{.*\}', metin, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


kayitlar = []
basla = time.perf_counter()

for i, s in enumerate(SORULAR, 1):
    ham = cevap_uret(s['soru'])
    veri = json_ayikla(ham)
    kayitlar.append({
        'soru': s['soru'],
        'stil': s['stil'],
        'beklenen_arac': s['arac'],
        'beklenen_par': s.get('parametreler', {}),
        'secilen_arac': (veri or {}).get('arac'),
        'secilen_par': (veri or {}).get('parametreler') or {},
        'sema_hatasi': veri is None,
        'ham': ham[:120],
    })
    print(f'{i:2d}/30', end='  ', flush=True)

gecen = time.perf_counter() - basla
print()
print('bitti:', round(gecen, 1), 'sn')


In [ ]:
def arac_dogru(k):
    return k['secilen_arac'] == k['beklenen_arac']


def par_dogru(k):
    secilen = {a: str(d).strip().casefold() for a, d in k['secilen_par'].items()}
    beklenen = k['beklenen_par']
    if set(secilen) != set(beklenen):
        return False
    return all(
        secilen[a] in {str(x).strip().casefold() for x in kabul}
        for a, kabul in beklenen.items()
    )


def uydurma(k):
    soru = k['soru'].casefold()
    return [
        f'{a}={d}' for a, d in k['secilen_par'].items()
        if str(d).strip() and str(d).strip().casefold() not in soru
    ]


n = len(kayitlar)
ad = sum(arac_dogru(k) for k in kayitlar)
td = sum(arac_dogru(k) and par_dogru(k) for k in kayitlar)
sema = sum(k['sema_hatasi'] for k in kayitlar)
uyd = sum(bool(uydurma(k)) for k in kayitlar)

secilenler = [k['secilen_arac'] for k in kayitlar if k['secilen_arac']]
cokus_arac, cokus_adet = Counter(secilenler).most_common(1)[0] if secilenler else ('', 0)
cokus_pay = cokus_adet / len(secilenler) if secilenler else 0

print('=' * 66)
print('1. TUR OLCUMU  (30 soru, B2.3 ile ayni set)')
print('=' * 66)
print(f"  arac dogrulugu    : %{ad / n * 100:5.1f}   (taban %{TABAN['arac_dogruluk'] * 100:.1f})")
print(f"  arac + parametre  : %{td / n * 100:5.1f}   (taban %{TABAN['tam_dogruluk'] * 100:.1f})")
print(f'  sema hatasi       : {sema}/30')
print(f'  uydurma parametre : {uyd}/30')
print(f'  cokus             : {cokus_arac} %{cokus_pay * 100:.0f}', '<-- COKUS!' if cokus_pay >= 0.40 else '(esik %40)')
print(f'  sure              : {round(gecen, 1)} sn')


In [ ]:
print('=' * 66)
print('ARAC BAZINDA        1. tur   taban')
print('-' * 66)
for arac in sorted({k['beklenen_arac'] for k in kayitlar}):
    ilgili = [k for k in kayitlar if k['beklenen_arac'] == arac]
    d = sum(arac_dogru(k) for k in ilgili)
    t = TABAN['arac_bazinda'].get(arac, [0, 0])
    isaret = ''
    if d > t[0]:
        isaret = '  ^ iyilesti'
    elif d < t[0]:
        isaret = '  v kotulesti'
    print(f'  {arac:32s} {d}/{len(ilgili)}     {t[0]}/{t[1]}{isaret}')

print()
print('STIL BAZINDA        1. tur   taban')
print('-' * 66)
for stil in ('acik', 'dolayli', 'gunluk'):
    ilgili = [k for k in kayitlar if k['stil'] == stil]
    if not ilgili:
        continue
    d = sum(arac_dogru(k) for k in ilgili)
    t = TABAN['stil_bazinda'].get(stil, [0, 0])
    print(f'  {stil:32s} {d}/{len(ilgili)}     {t[0]}/{t[1]}')


In [ ]:
print('=' * 66)
print('YANLIS CEVAPLAR')
print('=' * 66)
for k in kayitlar:
    if arac_dogru(k) and par_dogru(k):
        continue
    print()
    print('  SORU     :', k['soru'])
    print('  beklenen :', k['beklenen_arac'], k['beklenen_par'] or '')
    print('  secilen  :', k['secilen_arac'], k['secilen_par'] or '')
    if k['sema_hatasi']:
        print('  SEMA HATASI, ham cikti:', k['ham'])


---

# B3.3 · 2. tur — sınıf ağırlıklı örnekleme

## 1. turda ne ters gitti

Veri hazırlanırken router dosyasının **ilk 7.500 satırı** alınmıştı, rastgele
değil. Dosya araca göre sıralı olduğu için:

```
                                  1. TURDA    TUM DOSYA
  siparis_onerisi_sorgula            6.742       41.886   %90
  tedarikci_performansi_sorgula        543        1.325
  kritik_stok_sorgula                  116          256
  olu_stok_sorgula                      99          240
  gecelik_ozet_sorgula                   0           66   SIFIR
  onay_kuyrugu_sorgula                   0           14   SIFIR
  genel_stok_durumu_sorgula              0           11   SIFIR
```

Üç araç eğitime **hiç girmedi** ve model ölçümde o üçünü hiç seçmedi.
Router doğruluğu %70'ten %40'a düştü.

Eğitimin çalıştığının kanıtı da aynı tabloda: `olu_stok_sorgula` 99 örnek
gördü ve **1/5 → 4/5** çıktı.

## Düzeltme: her araca taban kota

`dengeli_kota()` eşit paya yakın dağıtır ama her sınıfı **kapasitesiyle**
sınırlar: `kapasite = mevcut × TAVAN_KAT`. Kapasitesi düşük sınıflar
kapasitesi kadar alır, artan bütçe diğerlerine dağıtılır.

### `TAVAN_KAT` neden var

11 örnekli bir aracı sınırsız tekrarlamak modele o **11 cümleyi ezberletir**,
aracı öğretmez. Tavan, tekrar katsayısını bağlar.

Yerelde ölçülen denge:

| tavan | özgün soru | en seyrek kota | dengesizlik |
|---|---|---|---|
| 10 | %47,5 | 110 | 15,0x |
| 20 | %44,4 | 220 | 6,5x |
| **40** | **%42,5** | **440** | **3,0x** |
| 80 | %37,2 | 880 | 1,3x |

Tavanı yükseltmek dengeyi çok ucuza alıyor (15x → 1,3x'e karşılık özgünlük
%47'den %37'ye). **40 seçildi**: dengesizlik 3x, özgünlük %42,5.

> ⚠️ **Hiçbir örnekleme 11 özgün soruyu çoğaltamaz.** Ölçüm iyileşebilir
> (model o 11 kalıbı öğrenir) ama aynı aracın *yeni* bir soruluşunu tanıması
> beklenmez. Gerçek çözüm seyrek araçlar için daha çeşitli soru üretmek —
> Kişi A'nın tarafı.

### Yeni veriyle yeniden ölçüldü

Kişi A'nın parafraz turundan sonra router havuzu **43.798 → 3.674** satıra indi
(dengeleme uygulandı) ve seyrek araçlar arttı: `genel_stok_durumu` 11→26,
`onay_kuyrugu` 14→39, `gecelik_ozet` 66→155 (eğitim bölümünde).

Havuz küçülünce 7.500 hedefi anlamsız hale geldi — her şeyi 2 kat tekrarlamak
olurdu. Yeniden ölçüldü:

| router hedefi | özgün | en çok tekrar | dengesizlik |
|---|---|---|---|
| 7.500 | %38,3 | 40,0x | 1,0x |
| 3.500 | %49,0 | 19,2x | 1,0x |
| **2.000** | **%64,5** | **11,0x** | **1,0x** |
| 1.400 | %72,9 | 7,7x | 1,0x |

Denge her ayarda tam; fark **özgünlükte**. Tekrarlanan örnek ilk birkaç
geçişten sonra neredeyse hiçbir şey öğretmiyor — düşük özgünlük boşa hesap.

**Seçilen: router 2.000, gerekçe 8.000.** Gerekçe tarafında 40 bin özgün örnek
var, orayı kısmaya gerek yok. Toplam 10.000 — Tur 1'den (15.000) küçük ama
üç aracın hiç görülmediği bir 15.000'den kesinlikle iyi.

⚠️ Yeni dosya da **araca göre sıralı**. İlk-N almak bu veriyle de aynı hatayı
üretirdi; dengeli örnekleme hâlâ zorunlu.


In [ ]:
import json
import random
from collections import Counter, defaultdict

# Yeni (dengelenmis) veriyle olculdu -- ayrinti yukaridaki markdown'da.
TAVAN_KAT = 20
ROUTER_ORNEK = 2_000
GEREKCE_ORNEK = 8_000


def dengeli_kota(mevcut, toplam, tavan_kat=TAVAN_KAT):
    """Esit paya yakin dagitir, her sinifi kapasitesiyle sinirlar."""
    kalan = dict(mevcut)
    kota = {}
    butce = toplam
    while kalan and butce > 0:
        pay = butce // len(kalan)
        if pay == 0:
            break
        doyan = {a: m * tavan_kat for a, m in kalan.items() if m * tavan_kat <= pay}
        if not doyan:
            for a in kalan:
                kota[a] = pay
            break
        for a, k in doyan.items():
            kota[a] = k
            butce -= k
            del kalan[a]
    return kota


def dengeli_ornekle(kayitlar, anahtar, toplam, tohum=42):
    rnd = random.Random(tohum)
    gruplar = defaultdict(list)
    for k in kayitlar:
        gruplar[k[anahtar]].append(k)
    kota = dengeli_kota({a: len(v) for a, v in gruplar.items()}, toplam)
    secilen = []
    for a, adet in kota.items():
        havuz = gruplar[a]
        if adet <= len(havuz):
            secilen += rnd.sample(havuz, adet)
        else:
            secilen += havuz * (adet // len(havuz)) + rnd.sample(havuz, adet % len(havuz))
    rnd.shuffle(secilen)
    return secilen


# ⚠️ TUM dosya okunuyor -- 1. turdaki hata buradaydi (ilk N satir aliniyordu)
with (VERI / 'router_train.jsonl').open(encoding='utf-8') as f:
    router_hepsi = [json.loads(s) for s in f]
with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
    gerekce_hepsi = [json.loads(s) for s in f]

print('okunan  router :', len(router_hepsi))
print('okunan  gerekce:', len(gerekce_hepsi))


In [ ]:
router_sec = dengeli_ornekle(router_hepsi, 'arac', ROUTER_ORNEK)
gerekce_sec = dengeli_ornekle(gerekce_hepsi, 'karar_tipi', GEREKCE_ORNEK)

print('ROUTER  arac dagilimi')
print('-' * 58)
mevcut = Counter(k['arac'] for k in router_hepsi)
yeni = Counter(k['arac'] for k in router_sec)
for a in sorted(mevcut):
    print(f'  {a:32s} {mevcut[a]:6d} -> {yeni[a]:5d}  ({yeni[a] / mevcut[a]:.1f}x)')
print(f'  disarida kalan arac: {sum(1 for a in mevcut if yeni[a] == 0)}')
print(f'  ozgun soru: {len({k[chr(115) + chr(111) + chr(114) + chr(117)] for k in router_sec})}/{len(router_sec)}')
print()
print('GEREKCE  karar tipi dagilimi')
print('-' * 58)
gm = Counter(k['karar_tipi'] for k in gerekce_hepsi)
gy = Counter(k['karar_tipi'] for k in gerekce_sec)
for t in sorted(gm):
    print(f'  {t:32s} {gm[t]:6d} -> {gy[t]:5d}')


In [ ]:
from datasets import Dataset

metinler = [gerekce_metni(k) for k in gerekce_sec] + [router_metni(k) for k in router_sec]
random.Random(42).shuffle(metinler)
metinler = [m + tokenizer.eos_token for m in metinler]

egitim = Dataset.from_dict({'text': metinler})
print(len(egitim), 'ornek hazir (2. tur)')
print('-' * 70)
print(egitim[0]['text'][:350])


## Buradan sonrası 1. turla aynı

Yukarıdaki **3. hücreden** (taban modeli sıfırdan yükle) devam et. Tek fark:
`egitim` artık dengeli örneklenmiş veri.

⚠️ `CIKIS` yolunu değiştir (`b33_tur2`), yoksa 1. turun checkpoint'lerinden
devam etmeye çalışır:


In [ ]:
CIKIS = str(CHECKPOINT / 'b33_tur2')
ARGS['output_dir'] = CIKIS
print('checkpoint dizini:', CIKIS)


---

# B3.4 · Merge → GGUF → Ollama

Eğitilmiş LoRA'yı taban modelle birleştirip Ollama'nın anlayacağı biçime
çevirir. Bundan sonra model **yerelde**, ERP'nin yanında çalışır.

## GPU gerekmiyor

Birleştirme ve dönüştürme işlemci işi. Kota dolmuşsa **"GPU olmadan bağlan"**
diyerek de koşturulabilir — sadece biraz daha yavaş olur.

## Neden `q8_0`

GGUF'a çevirirken nicemleme (quantization) seçiliyor:

| biçim | boyut | kalite |
|---|---|---|
| `f16` | ~3,1 GB | kayıpsız |
| **`q8_0`** | **~1,6 GB** | pratikte kayıpsız |
| `q4_k_m` | ~1,0 GB | hafif kayıp, ayrıca `llama-quantize` derlemesi ister |

`q8_0` seçildi: `convert_hf_to_gguf.py` doğrudan üretebiliyor (derleme yok),
1,5B modelde kalite farkı ölçülemeyecek kadar küçük, indirme 1,6 GB.

⚠️ **Bu bölüm 1. turun değil, 2. turun adaptörünü kullanmalı.** Aşağıdaki
`LORA_YOLU`'nu kontrol et — yanlış klasörü gösterirse yanlış modeli ölçeriz.


## 1. Ortam


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
!pip install -q peft transformers accelerate sentencepiece protobuf
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt


## 2. Hangi adaptör?

⚠️ Ölçümün doğru olması için **2. turun** adaptörü kullanılmalı. Klasörleri
listeleyip tarihine bakıyoruz.


In [ ]:
import os
from datetime import datetime
from pathlib import Path

CIKTI = Path('/content/drive/MyDrive/codifya/cikti')

for d in sorted(CIKTI.iterdir()):
    if not d.is_dir():
        continue
    agirlik = d / 'adapter_model.safetensors'
    if not agirlik.exists():
        print(f'  {d.name:24s} (adapter yok)')
        continue
    t = datetime.fromtimestamp(agirlik.stat().st_mtime)
    mb = agirlik.stat().st_size / 1024**2
    print(f'  {d.name:24s} {mb:7.1f} MB   {t:%Y-%m-%d %H:%M}')


In [ ]:
# Yukaridaki listeden 2. TURUN klasorunu sec.
LORA_YOLU = CIKTI / 'b33_tur1_lora'  # <-- 2. tur buraya kaydedildiyse dogru
MODEL_ETIKETI = 'tur2'  # olcumde kullanilacak ad

assert (LORA_YOLU / 'adapter_model.safetensors').exists(), 'adapter bulunamadi'
print('kullanilacak adapter:', LORA_YOLU)


## 3. LoRA'yı taban modelle birleştir

LoRA küçük ek matrisler tutuyor; Ollama tam bir model istiyor. Birleştirme
bu ek matrisleri taban ağırlıklara işliyor.

⚠️ 4-bit **kullanılmıyor**: birleştirme tam duyarlıkla yapılmalı, yoksa
nicemleme hatası kalıcı olarak ağırlıklara işlenir. Bellek ~4 GB, Colab yeterli.


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

TABAN = 'Qwen/Qwen2.5-1.5B-Instruct'
BIRLESIK = Path('/content/birlesik')

taban = AutoModelForCausalLM.from_pretrained(
    TABAN,
    torch_dtype=torch.float16,
    device_map='cpu',
)
tokenizer = AutoTokenizer.from_pretrained(TABAN)

model = PeftModel.from_pretrained(taban, str(LORA_YOLU))
model = model.merge_and_unload()

model.save_pretrained(str(BIRLESIK), safe_serialization=True)
tokenizer.save_pretrained(str(BIRLESIK))

toplam = sum(f.stat().st_size for f in BIRLESIK.rglob('*') if f.is_file())
print('birlestirildi:', BIRLESIK, f'({toplam / 1024**3:.2f} GB)')


## 4. GGUF'a çevir (`q8_0`)


In [ ]:
GGUF = f'/content/codifya-{MODEL_ETIKETI}-q8_0.gguf'

!python /content/llama.cpp/convert_hf_to_gguf.py /content/birlesik \
    --outfile {GGUF} --outtype q8_0

import os

print()
print('GGUF:', GGUF)
print('boyut:', round(os.path.getsize(GGUF) / 1024**3, 2), 'GB')


## 5. Drive'a kopyala ve indir

İki yol: Drive'a kopyalayıp oradan indirmek (kesintiye dayanıklı) ya da
doğrudan tarayıcıdan indirmek. 1,6 GB için Drive daha güvenli.


In [ ]:
import shutil

hedef = CIKTI / os.path.basename(GGUF)
shutil.copy(GGUF, hedef)
print('Drive\'a kopyalandi:', hedef)
print('Simdi Drive arayuzunden indir, sonra yerelde:')
print()
print('  ollama create codifya-router:' + MODEL_ETIKETI + ' -f Modelfile')


## 6. Yerelde Ollama'ya tanıt

GGUF indikten sonra **yerelde** (Colab'da değil) çalıştırılacak. `Modelfile`
repoda: `training/Modelfile`.

### ⚠️ En kritik ayar: `TEMPLATE`

Ollama varsayılan olarak modelin **sohbet şablonunu** uygular
(`<|im_start|>user ...`). Ama bu model **ham metinle** eğitildi:

```
GOREV: router
SORU: kritik stok var mi

ARAC:
```

Sohbet şablonu uygulanırsa model eğitimde hiç görmediği bir sarmalayıcı
görür ve eğitimin kazandırdığı kaybolur. `Modelfile`'daki
`TEMPLATE """{{ .Prompt }}"""` bunu engelliyor — istem olduğu gibi geçiyor.

Kurulum:

```bash
# indirilen GGUF'u training/ klasorune koy, sonra:
ollama create codifya-router:tur2 -f training/Modelfile
ollama list
```

### Sonra ölçüm

```bash
uv run python -m training.eval.router_taban \
  --model codifya-router:tur2 --istem-bicimi egitilmis --etiket lora-tur2
```

Aynı 30 soru, aynı puanlama, aynı şema kısıtı, aynı sıcaklık ve tohum.
Tek değişen model. Taban çizgi: **araç %70,0 · tam %66,7**.
